## 1. Load Data

In [1]:
import pandas as pd
import torch
import numpy as np
import unicodedata
from transformers import AutoTokenizer, AutoModel
from collections import defaultdict

print("Loading datasets...")
word_level_df = pd.read_csv('../data/Amirim_Project_Submission/translated_podcast_transcript_filtered.csv')

with open("../data/podcast_sentences.csv", "r", encoding="utf-8") as f:
    lines = f.readlines()
sentences = [line.strip().split(',', 1)[1] for line in lines[1:] if ',' in line]

print(f"Target words : {len(word_level_df)}")
print(f"Sentences    : {len(sentences)}")

/Users/YAHLIZ/miniforge3/envs/language_project_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading datasets...
Target words : 1735
Sentences    : 402


## 2. Helpers

In [2]:
def normalize(text):
    text = unicodedata.normalize('NFC', str(text))
    return text.strip(' .,!?"\'()-:;[]{}').lower()

def sentence_contains_components(sentence_lower, components):
    """Check if all phrase components appear consecutively in the sentence."""
    words = sentence_lower.split()
    words_norm = [normalize(w) for w in words]
    for i in range(len(words_norm) - len(components) + 1):
        if all(words_norm[i+j] == components[j] for j in range(len(components))):
            return True
    return False

def get_sentence_tokens(sentence, tokenizer, model):
    encoded = tokenizer(sentence, return_tensors='pt', truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**encoded)
    token_embeddings = outputs.last_hidden_state.squeeze(0)
    word_ids = encoded.word_ids()
    input_ids = encoded['input_ids'][0]

    word_vectors = {}
    word_token_ids = {}
    for idx, word_id in enumerate(word_ids):
        if word_id is None:
            continue
        if word_id not in word_vectors:
            word_vectors[word_id] = []
            word_token_ids[word_id] = []
        word_vectors[word_id].append(token_embeddings[idx].numpy())
        word_token_ids[word_id].append(input_ids[idx].item())

    tokens = []
    for word_id in sorted(word_vectors.keys()):
        avg_vector = np.mean(word_vectors[word_id], axis=0)
        raw_text = tokenizer.decode(word_token_ids[word_id])
        norm_text = normalize(raw_text)
        if norm_text:
            tokens.append({"text": norm_text, "vector": avg_vector})
    return tokens

## 3. Assign Words to Sentences

In [3]:
print("Assigning target words to sentences...")

word_to_sentence = {}
unassignable = []

sent_idx = 0
for wi in range(len(word_level_df)):
    target_raw = str(word_level_df.iloc[wi]['en']).strip().lower()
    components = [c.strip() for c in target_raw.split('_')]

    found = False
    for lookahead in range(min(6, len(sentences) - sent_idx)):
        candidate_idx = sent_idx + lookahead
        if sentence_contains_components(sentences[candidate_idx].lower(), components):
            word_to_sentence[wi] = candidate_idx
            sent_idx = candidate_idx
            found = True
            break

    if not found:
        unassignable.append(wi)
        word_to_sentence[wi] = None

assigned = sum(1 for v in word_to_sentence.values() if v is not None)
print(f"Assigned : {assigned} / {len(word_level_df)}")
print(f"Dropped  : {len(unassignable)}")

print("\nFirst 10 assignments:")
for wi in range(10):
    si = word_to_sentence[wi]
    sent_preview = sentences[si][:70] if si is not None else "UNASSIGNED"
    print(f"  [{wi:3d}] '{word_level_df.iloc[wi]['en']}'  -> sent {si}: '{sent_preview}'")

Assigning target words to sentences...
Assigned : 1526 / 1735
Dropped  : 209

First 20 assignments:
  [  0] 'act'  -> sent 0: 'Act One, Monkey in the Middle.'
  [  1] 'monkey'  -> sent 0: 'Act One, Monkey in the Middle.'
  [  2] 'middle'  -> sent 0: 'Act One, Monkey in the Middle.'
  [  3] 'places'  -> sent 1: 'So there's some places where animals almost never go, places that are '
  [  4] 'animals'  -> sent 1: 'So there's some places where animals almost never go, places that are '
  [  5] 'go'  -> sent 1: 'So there's some places where animals almost never go, places that are '
  [  6] 'places'  -> sent 1: 'So there's some places where animals almost never go, places that are '
  [  7] 'designed'  -> sent 1: 'So there's some places where animals almost never go, places that are '
  [  8] 'humans'  -> sent 1: 'So there's some places where animals almost never go, places that are '
  [  9] 'humans'  -> sent 1: 'So there's some places where animals almost never go, places that are '
  [ 

## 4. Load Model

In [4]:
print("Loading XLM-RoBERTa...")
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
model = AutoModel.from_pretrained('xlm-roberta-base')
model.eval()

Loading XLM-RoBERTa...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]Error processing line 1 of /Users/YAHLIZ/miniforge3/envs/language_project_env/lib/python3.10/site-packages/distutils-precedence.pth:

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14403.72it/s]
  Traceback (most recent call last):
    File "/Users/YAHLIZ/miniforge3/envs/language_project_env/lib/python3.10/site.py", line 195, in addpackage
      exec(line)
    File "<string>", line 1, in <module>
  ModuleNotFoundError: No module named '_distutils_hack'

Remainder of file ignored
XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; n

XLMRobertaModel(
  (embeddings): XLMRobertaEmbeddings(
    (word_embeddings): Embedding(250002, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
  )
  (encoder): XLMRobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x XLMRobertaLayer(
        (attention): XLMRobertaAttention(
          (self): XLMRobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): XLMRobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=Tru

## 5. Extraction Loop

In [5]:
print("Extracting embeddings...\n")

final_aligned_embeddings = []
matched_word_indices = []
dropped_word_indices = []

words_per_sentence = defaultdict(list)
for wi, si in word_to_sentence.items():
    if si is not None:
        words_per_sentence[si].append(wi)

for si in sorted(words_per_sentence.keys()):
    sentence = sentences[si]
    target_word_indices = sorted(words_per_sentence[si])
    sentence_tokens = get_sentence_tokens(sentence, tokenizer, model)

    sentence_pointer = 0
    for word_idx in target_word_indices:
        target_raw = str(word_level_df.iloc[word_idx]['en']).strip().lower()
        target_components = [c.strip() for c in target_raw.split('_')]
        phrase_len = len(target_components)

        matched = False
        search_start = sentence_pointer
        while search_start <= len(sentence_tokens) - phrase_len:
            if all(sentence_tokens[search_start + i]['text'] == target_components[i]
                   for i in range(phrase_len)):
                phrase_vectors = [sentence_tokens[search_start + i]['vector']
                                  for i in range(phrase_len)]
                final_aligned_embeddings.append(np.mean(phrase_vectors, axis=0))
                matched_word_indices.append(word_idx)
                sentence_pointer = search_start + phrase_len
                matched = True
                break
            search_start += 1

        if not matched:
            dropped_word_indices.append(word_idx)

Extracting embeddings...



## 6. Report

In [6]:
print("=" * 55)
print("FINAL ALIGNMENT REPORT")
print("=" * 55)
print(f"Total target words  : {len(word_level_df)}")
print(f"Successfully matched: {len(final_aligned_embeddings)}")
print(f"Dropped             : {len(dropped_word_indices)}")
print(f"Match rate          : {len(final_aligned_embeddings)/len(word_level_df)*100:.1f}%")

if dropped_word_indices:
    print(f"\nDropped words (first 20):")
    for idx in dropped_word_indices[:20]:
        row = word_level_df.iloc[idx]
        si = word_to_sentence.get(idx)
        print(f"  [{idx:4d}] '{row['en']}'  -> sent {si}: '{sentences[si][:60] if si else 'NONE'}'")

FINAL ALIGNMENT REPORT
Total target words  : 1735
Successfully matched: 1441
Dropped             : 85
Match rate          : 83.1%

Dropped words (first 20):
  [  46] 'monkey'  -> sent 9: 'And it was just so amazing just to be like a monkey.'
  [  53] 'know'  -> sent 12: 'And I know a little bit about monkey etiquette and showing y'
  [  56] 'monkey'  -> sent 12: 'And I know a little bit about monkey etiquette and showing y'
  [  99] 'curious'  -> sent 18: 'He was hoping they'd get curious about the camera and start '
  [ 100] 'pushing'  -> sent 18: 'He was hoping they'd get curious about the camera and start '
  [ 157] 'monkeys'  -> sent 34: 'A few other monkeys noticed their strange new friend lying f'
  [ 164] 'monkeys'  -> sent 35: 'So I had monkeys on my back whilst I'm trying to keep my cam'
  [ 165] 'trying'  -> sent 35: 'So I had monkeys on my back whilst I'm trying to keep my cam'
  [ 262] 'agents'  -> sent 54: 'And I just thought, I need to get this to my agents.'
  [ 329] 'wa

## 7. Save

In [8]:
if len(final_aligned_embeddings) > 0:
    embeddings_df = pd.DataFrame(final_aligned_embeddings)
    out_emb = '../data/processed/en_contextual_aligned_embeddings.csv'
    out_idx = '../data/processed/en_contextual_matched_indices.csv'
    embeddings_df.to_csv(out_emb, index=False)
    pd.DataFrame({'original_word_idx': matched_word_indices}).to_csv(out_idx, index=False)
    print(f"Saved embeddings -> {out_emb}  shape: {embeddings_df.shape}")
    print(f"Saved index map  -> {out_idx}")

    emb = pd.read_csv('../data/processed/en_contextual_aligned_embeddings.csv')
    idx = pd.read_csv('../data/processed/en_contextual_matched_indices.csv')

    print(f"Embeddings shape : {emb.shape}")       # should be (1441, 768)
    print(f"Index map shape  : {idx.shape}")        # should be (1441, 1)
    print(f"Index range      : {idx['original_word_idx'].min()} - {idx['original_word_idx'].max()}")
    print(f"Any NaN in embeddings: {emb.isnull().any().any()}")
    print(f"\nFirst 5 indices: {idx['original_word_idx'].tolist()[:5]}")
    print(f"Sample embedding row 0 (first 5 dims): {emb.iloc[0, :5].tolist()}")

Saved embeddings -> ../data/processed/en_contextual_aligned_embeddings.csv  shape: (1441, 768)
Saved index map  -> ../data/processed/en_contextual_matched_indices.csv
Embeddings shape : (1441, 768)
Index map shape  : (1441, 1)
Index range      : 0 - 1734
Any NaN in embeddings: False

First 5 indices: [0, 1, 2, 3, 4]
Sample embedding row 0 (first 5 dims): [0.042788513, 0.047795862, 0.008115685, 0.005766019, 0.022064418]
